# Identity-panel experiments on a Kaggle GPU (open-weights model, no API)
Runs **all 3 tests / 8 conditions** on a model downloaded from Hugging Face. Progress is kept in a **Hugging Face dataset**, so every Kaggle session resumes where the last one stopped and your weekly GPU hours add up across sessions.

**One-time setup**
1. Right sidebar -> *Session options*: **Accelerator = GPU T4 x2** (a 7B model in fp16 needs ~15 GB, so it is split over two T4s; on a single 16 GB GPU use a 3B model or set `LOAD_IN_4BIT = True`). *Internet = On* (needed to download the weights; Kaggle may ask you to verify your phone).
2. *Add-ons -> Secrets*: add `HF_TOKEN` (Hugging Face token with **write** access) and tick *Attached*.
3. Edit the next cell, then **Save Version -> Save & Run All (Commit)** so it keeps running in the background after you close the tab.

Run **one session at a time**. The session stops itself cleanly after `MAX_MINUTES` and uploads everything - re-run the notebook for the next chunk. Check your remaining GPU hours in your Kaggle profile (Settings).

In [ ]:
HF_REPO      = "YOUR_HF_USERNAME/identity-panel-results"   # private dataset repo; created automatically
CODE_URL     = "https://github.com/YOUR_GITHUB_USERNAME/aditi-dsa.git"
MODEL        = "Qwen/Qwen2.5-7B-Instruct"      # any chat model on the Hub; 3B example: "Qwen/Qwen2.5-3B-Instruct"
LOAD_IN_4BIT = False                           # True = less GPU memory (needs bitsandbytes), usually slower
N_SCENARIOS  = 300      # how many scenarios to pick AT RANDOM from popquorn.jsonl; None = all of them
SEED         = 0        # same seed -> same scenarios (and a bigger N later keeps the earlier ones)
ROUNDS       = 2        # discussion rounds in the discussion conditions
MAX_MINUTES  = 600      # stop cleanly after this long (Kaggle sessions are capped around 12 h)
VERIFY_FIRST = True     # first check that the biased judge really behaves differently (a few minutes)

In [ ]:
import os, subprocess, sys
CODE_DIR = "/kaggle/working/aditi-dsa"        # or the path of a Kaggle Dataset holding the code
if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", CODE_URL, CODE_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate",
                "huggingface_hub", "bitsandbytes"], check=True)
import torch
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] or "NONE - set Accelerator to GPU!")

In [ ]:
# The model is downloaded here (and cached for this session). HF_TOKEN is only needed for gated models.
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

common = ["--items", f"{CODE_DIR}/examples/popquorn.jsonl", "--model", MODEL, "--seed", str(SEED),
          "--rounds", str(ROUNDS), "--workers", "32", "--max-batch", "16"]
if LOAD_IN_4BIT:
    common.append("--load-in-4bit")

if VERIFY_FIRST:
    rc = subprocess.run([sys.executable, "-u", f"{CODE_DIR}/scripts/run_suite.py", *common,
                         "--verify-bias", "--verify-n", "8", "--out", "/kaggle/working/verify.jsonl"]).returncode
    assert rc == 0, ("The biased judge did not behave differently from a normal one with this model - "
                     "see the output above. Try a stronger model, or set VERIFY_FIRST = False to run anyway.")

In [ ]:
args = ["--hf-repo", HF_REPO, "--max-runtime-minutes", str(MAX_MINUTES), "--", *common]
if N_SCENARIOS is not None:
    args += ["--n-scenarios", str(N_SCENARIOS)]
rc = subprocess.run([sys.executable, "-u", f"{CODE_DIR}/scripts/kaggle_runner.py", *args]).returncode
print("exit code:", rc, "(0 = done, or stopped cleanly at the time cap -> run the notebook again to continue)")

Results are in your Hugging Face dataset: `results.jsonl` (every judge's score and reasoning, every round, every condition), `results_summary.csv` (mean bias gap per condition), and `results.cache.jsonl` (needed to resume - keep it).